# Data audit: what the data can and cannot measure

**Purpose.** This notebook records how the assumptions of the project *What happens to professional football players after a serious meniscal injury in Europe's top five leagues?* were checked against the data before any threshold, window or definition was fixed.

**Data (Transfermarkt, via Kaggle).**
- dataset 1, `xfkzujqjvx97n/football-datasets`: `player_injuries`, `player_performances`
- dataset 2, `davidcariboo/player-scores`: `players`, `appearances`, `games`, `transfers`

**Table of contents**

| Step | Question |
|---|---|
| 0 | Which tables do we have, and what do their keys, missing values and date ranges look like? |
| 1 | For meniscal injuries, how many have data before and after the injury? |
| 2 | Do the two datasets agree where they describe the same thing? |
| 3 | From when to when are the data complete? |
| 4 | How good are the meniscal injury records? |
| 5 | How is absence distributed, and where is the severity threshold? |
| 6 | Which competition levels are visible in each table, and how do we label the level after return? |
| 7 | Which role measure and which minimum minutes baseline? |
| 8 | How many players vanish after return, and can a club change be detected? |

## Step 0: What do we have? Tables, keys, missing values, date ranges

**Question.** Which tables do we have, how many rows, which column types and missing values do they have, are the keys unique where they should be, and which period and competitions do the tables cover?

### 0.1 Load the tables and 0.2 convert dates

In the CSV files dates are text. We convert them to dates and **count how many values failed to convert** (`errors="coerce"` silently turns them into missing values, so without this check the errors would disappear).

In [35]:
# 0.1 Setup and loading 
import numpy as np
import pandas as pd
from pathlib import Path

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

KAGGLE_A = Path("/kaggle/input/datasets/xfkzujqjvx97n/football-datasets")
KAGGLE_B = Path("/kaggle/input/datasets/davidcariboo/player-scores")

def csv_path(name):
    candidates = [KAGGLE_A / name / f"{name}.csv", KAGGLE_B / f"{name}.csv",
                  Path("data") / f"{name}.csv", Path("../data") / f"{name}.csv"]
    found = next((p for p in candidates if p.exists()), None)
    if found is None:
        raise FileNotFoundError(f"{name}.csv not found; looked in: {[str(p) for p in candidates]}")
    return found

player_injuries     = pd.read_csv(csv_path("player_injuries"))
player_performances = pd.read_csv(csv_path("player_performances"))
players             = pd.read_csv(csv_path("players"))
appearances         = pd.read_csv(csv_path("appearances"))
games               = pd.read_csv(csv_path("games"))
transfers           = pd.read_csv(csv_path("transfers"))

tables = {"player_injuries": player_injuries, "player_performances": player_performances,
          "players": players, "appearances": appearances, "games": games, "transfers": transfers}
for name, df in tables.items():
    print(f"{name:20s} rows: {len(df):>9,}   columns: {df.shape[1]}")

# 0.2 Dates: conversion and range 
date_cols = {"player_injuries": ["from_date", "end_date"], "appearances": ["date"], "games": ["date"],
             "transfers": ["transfer_date"], "players": ["date_of_birth"]}
rows = []
for name, cols in date_cols.items():
    df = tables[name]          # the same object as the variable of that name, so the change is visible everywhere
    for c in cols:
        before = df[c].isna().sum()
        df[c] = pd.to_datetime(df[c], errors="coerce")
        rows.append({"table": name, "column": c, "missing before": before,
                     "new missing after conversion": df[c].isna().sum() - before,
                     "earliest": df[c].min(), "latest": df[c].max()})
pd.DataFrame(rows)

player_injuries      rows:   143,195   columns: 7
player_performances  rows: 1,878,719   columns: 20
players              rows:    50,149   columns: 26
appearances          rows: 1,894,350   columns: 13
games                rows:    88,958   columns: 23
transfers            rows:   175,165   columns: 10


,table,column,missing before,new missing after conversion,earliest,latest
0,player_injuries,from_date,22,0,1973-09-02,2025-12-22
1,player_injuries,end_date,1523,0,1973-09-06,2026-09-01
2,appearances,date,0,0,2012-07-03,2026-06-28
3,games,date,0,0,2006-06-09,2026-07-06
4,transfers,transfer_date,0,0,1993-07-01,2030-06-30
5,players,date_of_birth,49,0,1968-07-31,2011-02-23


### 0.3 Columns and 0.4 keys

For every table: column type, share of missing values and number of unique values. Then: are the identifiers unique where they should be, and how many distinct players does each table contain?

In [36]:
# 0.3 Columns: type, missing values, unique values 
for name, df in tables.items():
    print("=" * 90)
    print(name, df.shape)
    display(pd.DataFrame({"type": df.dtypes.astype(str),
                          "missing": df.isna().sum(),
                          "missing %": (df.isna().mean() * 100).round(1),
                          "unique": df.nunique()}))

# 0.4 Keys and players per table 
print("players:      duplicate player_id     =", players["player_id"].duplicated().sum())
print("games:        duplicate game_id       =", games["game_id"].duplicated().sum())
print("appearances:  duplicate appearance_id =", appearances["appearance_id"].duplicated().sum())
print("transfers:    duplicate (player_id, transfer_date, from_club_id, to_club_id) =",
      transfers.duplicated(["player_id", "transfer_date", "from_club_id", "to_club_id"]).sum())
print("player_performances: duplicate (player_id, season_name, competition_id, team_id) =",
      player_performances.duplicated(["player_id", "season_name", "competition_id", "team_id"]).sum())
print("player_injuries: fully identical rows        =", player_injuries.duplicated().sum())

pd.Series({n: tables[n]["player_id"].nunique() for n in
           ["player_injuries", "player_performances", "players", "appearances", "transfers"]}, name="unique players")

player_injuries (143195, 7)


,type,missing,missing %,unique
player_id,int64,0,0.0,34561
season_name,str,0,0.0,46
injury_reason,str,0,0.0,349
from_date,datetime64[us],22,0.0,8133
end_date,datetime64[us],1523,1.1,8114
days_missed,float64,22,0.0,866
games_missed,int64,0,0.0,135


player_performances (1878719, 20)


,type,missing,missing %,unique
player_id,int64,0,0.0,88375
season_name,str,0,0.0,102
competition_id,str,0,0.0,1643
competition_name,str,0,0.0,1988
team_id,int64,0,0.0,13511
team_name,str,0,0.0,15411
nb_in_group,int64,0,0.0,51
nb_on_pitch,int64,0,0.0,52
goals,float64,138202,7.4,48
assists,int64,0,0.0,29


players (50149, 26)


...

,type,missing,missing %,unique
game_id,int64,0,0.0,88958
competition_id,str,0,0.0,70
season,int64,0,0.0,18
round,str,0,0.0,128
date,datetime64[us],0,0.0,4445
home_club_id,int64,0,0.0,2980
away_club_id,int64,0,0.0,2634
home_club_goals,int64,0,0.0,16
away_club_goals,int64,0,0.0,18
home_club_position,float64,25377,28.5,21


transfers (175165, 10)


,type,missing,missing %,unique
player_id,int64,0,0.0,23379
transfer_date,datetime64[us],0,0.0,5634
transfer_season,str,0,0.0,35
from_club_id,int64,0,0.0,17408
to_club_id,int64,0,0.0,13208
from_club_name,str,0,0.0,18285
to_club_name,str,0,0.0,14086
transfer_fee,float64,61526,35.1,1503
market_value_in_eur,float64,68160,38.9,191
player_name,str,0,0.0,23036


players:      duplicate player_id     = 0
games:        duplicate game_id       = 0
appearances:  duplicate appearance_id = 0
transfers:    duplicate (player_id, transfer_date, from_club_id, to_club_id) = 0
player_performances: duplicate (player_id, season_name, competition_id, team_id) = 0
player_injuries: fully identical rows        = 111


player_injuries        34561
player_performances    88375
players                50149
appearances            29531
transfers              23379
Name: unique players, dtype: int64

### 0.5 Competition coverage in `appearances`

`games` lists matches of many competitions, but `appearances` (one row per player per match) may cover only some of them. We check which competitions never appear in `appearances` and  what share of matches has no `appearances` rows at all, for the top-5 leagues separately from the rest, within the period that `appearances` covers.

In [37]:
# 0.5 Competition coverage in `appearances` -----------------------------------
TOP5 = ["GB1", "ES1", "IT1", "L1", "FR1"]     # Premier League, La Liga, Serie A, Bundesliga, Ligue 1

comp_type = games.drop_duplicates("competition_id").set_index("competition_id")["competition_type"]
in_app = set(appearances["competition_id"])
never = sorted(set(comp_type.index) - in_app)
print("competitions in games:", len(comp_type), "| in appearances:", len(in_app), "| in games but never in appearances:", len(never))
print("never in appearances:", never)

g = games[games["date"] >= appearances["date"].min()].copy()      # only the period that appearances covers
g["has_rows"] = g["game_id"].isin(set(appearances["game_id"]))
g["group"] = np.where(g["competition_id"].isin(TOP5), "top-5 leagues",
                      np.where(g["competition_type"] == "domestic_league", "other domestic leagues", g["competition_type"].astype(str)))
cov = g.groupby("group")["has_rows"].agg(games="size", with_rows="sum")
cov["% games without any row"] = (100 * (1 - cov["with_rows"] / cov["games"])).round(1)
display(cov)

competitions in games: 70 | in appearances: 48 | in games but never in appearances: 22
never in appearances: ['A1', 'AFAC', 'ARG1', 'AUS1', 'BRA1', 'C1', 'COL1', 'COPA', 'EURO', 'JAP1', 'KR1', 'MEX1', 'MLS1', 'NO1', 'PL1', 'RO1', 'RSK1', 'SA1', 'SE1', 'SER1', 'TS1', 'UCOL']


,games,with_rows,% games without any row
group,,,
domestic_cup,12904,7870,39.0
international_cup,3117,2872,7.9
national_team_competition,552,124,77.5
other,7599,5671,25.4
other domestic leagues,38141,30810,19.2
top-5 leagues,25241,25239,0.0


### 0.6 Two crucial details of `player_performances`
We check two details of `player_performances` that matter later: how seasons are written, and how often `minutes_played` is missing.

In [38]:
# 0.6 Two details of player_performances --------------------------------------
pp = player_performances
season_format = np.select([pp["season_name"].str.match(r"^\d{2}/\d{2}$"), pp["season_name"].str.match(r"^\d{4}$")],
                          ["two-year, e.g. 19/20", "single year, e.g. 2019"], default="other")
display(pd.Series(season_format).value_counts().rename("rows").to_frame())
print(f"minutes_played missing in player_performances: {pp['minutes_played'].isna().mean():.1%} of rows")

,rows
"two-year, e.g. 19/20",1546852
"single year, e.g. 2019",331867


minutes_played missing in player_performances: 62.3% of rows


### Findings

- Six tables; the largest are `player_performances` and `appearances` (about 1.9 million rows each). All date columns converted without losing a single value; `end_date` is missing in 1.1% of injury rows (`from_date` in 22 rows).
- Keys are unique (`player_id`, `game_id`, `appearance_id` and the combined keys in `transfers` and `player_performances`), with one exception: `player_injuries` has 111 rows identical in every column to another row.
- `appearances` covers 48 of the 70 competitions in `games` and is almost complete only for the top-5 leagues (2 of 25,241 matches missing). Other domestic leagues miss 19.2% of matches, domestic cups 39.0% and national teams 77.5%.
- In `player_performances` seasons are written in two ways ("19/20" and "2019"), and `minutes_played` is missing in 62.3% of rows.

## Step 1: Who has data before and after the injury?

**Question.** For meniscal injuries: how many players appear in each table, and for how many injuries do we have data in the window *before* and the window *after* the injury?


### 1.1 Which rows are meniscal injuries?

In [39]:
# every injury label that contains "menisc" (any capitalisation)
mask_menisc = player_injuries["injury_reason"].str.contains("menisc", case=False, na=False)
player_injuries.loc[mask_menisc, "injury_reason"].value_counts()

injury_reason
Meniscus injury                 942
Meniscus tear                   451
Meniscus damage                 293
Tear of the lateral meniscus    144
Meniscus irritation              33
Name: count, dtype: int64

In [40]:
MENISCUS_LABELS = ["Meniscus damage", "Meniscus tear", "Meniscus injury",
                   "Tear of the lateral meniscus", "Meniscus irritation"]

meniscus = player_injuries[player_injuries["injury_reason"].isin(MENISCUS_LABELS)]

print("meniscal injuries (rows):", len(meniscus))
print("players:", meniscus["player_id"].nunique())


meniscal injuries (rows): 1863
players: 1657


In [41]:
print("does the label list catch everything that contains 'menisc'?", len(meniscus) == mask_menisc.sum())

does the label list catch everything that contains 'menisc'? True


### 1.2 In which tables do these players appear at all?

Here we only ask whether a `player_id` exists in a table (ever, with no time windows).

In [42]:
ids = meniscus["player_id"].unique()

pd.Series({
    "players with a meniscal injury": len(ids),
    "...in players":                  np.isin(ids, players["player_id"]).sum(),
    "...in appearances":              np.isin(ids, appearances["player_id"]).sum(),
    "...in player_performances":      np.isin(ids, player_performances["player_id"]).sum(),
}, name="number of players")

players with a meniscal injury    1657
...in players                     1027
...in appearances                  845
...in player_performances         1657
Name: number of players, dtype: int64

### 1.3 Clean the injuries and build the windows

We drop injuries without a start or end date (no windows can be built for them) and full duplicates, and count how many that was. Every injury gets an `inj_id` so that results can be joined later.

In [43]:
KEY = ["player_id", "from_date", "end_date", "injury_reason"]

no_dates = meniscus[["from_date", "end_date"]].isna().any(axis=1).sum()
with_dates = meniscus.dropna(subset=["from_date", "end_date"])
duplicates = with_dates.duplicated(KEY).sum()
print("injuries without a start or end date:", no_dates)
print("full duplicates:", duplicates)

m = with_dates.drop_duplicates(KEY).reset_index(drop=True)
m["inj_id"] = m.index
print("injuries kept for the analysis:", len(m))

injuries without a start or end date: 25
full duplicates: 0
injuries kept for the analysis: 1838


In [44]:
PRE_MONTHS = 12
POST_MONTHS = 24

m["pre_start"] = m["from_date"] - pd.DateOffset(months=PRE_MONTHS)
m["post_end"] = m["end_date"] + pd.DateOffset(months=POST_MONTHS)

m[["player_id", "from_date", "end_date", "pre_start", "post_end"]].head()

,player_id,from_date,end_date,pre_start,post_end
0,100127,2012-01-10,2012-03-10,2011-01-10,2014-03-10
1,10076,2014-10-17,2015-02-25,2013-10-17,2017-02-25
2,100988,2022-10-05,2023-08-06,2021-10-05,2025-08-06
3,101253,2018-03-18,2018-09-18,2017-03-18,2020-09-18
4,10197,1998-11-25,1999-01-03,1997-11-25,2001-01-03


### 1.4 `appearances`: does the player have a match in the window before and in the window after?

For every injury we take all matches of that player (`merge` on `player_id`) and keep those that fall into the window. The injury gets `True` if at least one match remains.

In [45]:
a = appearances.loc[appearances["player_id"].isin(m["player_id"]), ["player_id", "date"]]

x = m[["inj_id", "player_id", "pre_start", "from_date", "end_date", "post_end"]].merge(a, on="player_id")

in_pre = (x["date"] >= x["pre_start"]) & (x["date"] < x["from_date"])
in_post = (x["date"] > x["end_date"]) & (x["date"] <= x["post_end"])

m["app_pre"] = m["inj_id"].isin(x.loc[in_pre, "inj_id"])
m["app_post"] = m["inj_id"].isin(x.loc[in_post, "inj_id"])

print("injuries:", len(m))
pd.DataFrame({"n": [m["app_pre"].sum(), m["app_post"].sum()],
              "%": [round(100 * m["app_pre"].mean(), 1), round(100 * m["app_post"].mean(), 1)]},
             index=["appearances: match BEFORE", "appearances: match AFTER"])

injuries: 1838


,n,%
appearances: match BEFORE,470,25.6
appearances: match AFTER,565,30.7


### 1.5 `player_performances`: seasons

In this table a season is written in two ways: `"19/20"` and `"2019"`. We turn both into **the year in which the season starts** (`"19/20"` → 2019, `"2019"` → 2019). For dates, the European season runs from July to June, so a date in month ≥ 7 belongs to the season numbered by its year, and an earlier date to the season numbered by year − 1.

In [46]:
def season_start(name):
    name = str(name)
    if len(name) == 4:                      # written as "2019"
        return int(name)
    yy = int(name[:2])                      # written as "19/20"
    return 1900 + yy if yy >= 50 else 2000 + yy

season_map = {s: season_start(s) for s in player_performances["season_name"].unique()}
player_performances["season_start"] = player_performances["season_name"].map(season_map)

def european_season(dates):
    return np.where(dates.dt.month >= 7, dates.dt.year, dates.dt.year - 1)

m["inj_season"] = european_season(m["from_date"])      # season in which the injury happened
m["ret_season"] = european_season(m["end_date"])       # season in which the player returned

# quick check: a few season names and their numbers
pd.Series(season_map).sort_values().iloc[[0, 1, 2, 50, 70, -3, -2, -1]]

1960     1960
60/61    1960
61/62    1961
2000     2000
10/11    2010
2025     2025
25/26    2025
2026     2026
dtype: int64

In [47]:
p = player_performances.loc[player_performances["player_id"].isin(m["player_id"]),
                            ["player_id", "season_start", "minutes_played"]]

y = m[["inj_id", "player_id", "inj_season", "ret_season"]].merge(p, on="player_id")

pre_season = y["season_start"] == y["inj_season"] - 1
post_seasons = (y["season_start"] > y["ret_season"]) & (y["season_start"] <= y["ret_season"] + 2)
has_minutes = y["minutes_played"].notna()

m["perf_pre"] = m["inj_id"].isin(y.loc[pre_season, "inj_id"])
m["perf_post"] = m["inj_id"].isin(y.loc[post_seasons, "inj_id"])
m["perf_pre_min"] = m["inj_id"].isin(y.loc[pre_season & has_minutes, "inj_id"])
m["perf_post_min"] = m["inj_id"].isin(y.loc[post_seasons & has_minutes, "inj_id"])

flags = {"performances: season row BEFORE":       "perf_pre",
         "performances: season row AFTER":        "perf_post",
         "performances with minutes: BEFORE":     "perf_pre_min",
         "performances with minutes: AFTER":      "perf_post_min"}
pd.DataFrame({"n": [m[c].sum() for c in flags.values()],
              "%": [round(100 * m[c].mean(), 1) for c in flags.values()]}, index=list(flags.keys()))

,n,%
performances: season row BEFORE,1755,95.5
performances: season row AFTER,1679,91.3
performances with minutes: BEFORE,1277,69.5
performances with minutes: AFTER,1324,72.0


### 1.6 Coverage table

The windows extend beyond the date range of `appearances` (2012-07 to 2026-06), so for some injuries data **cannot** exist for reasons of dates alone. The coverage table therefore uses only injuries whose two windows fit inside the `appearances` period . We also count how many injuries would fit if only 12 months after the return were required .

In [48]:
app_min, app_max = appearances["date"].min(), appearances["date"].max()
m["in_range"] = (m["pre_start"] >= app_min) & (m["post_end"] <= app_max)
m["in_range_12"] = (m["pre_start"] >= app_min) & (m["end_date"] + pd.DateOffset(months=12) <= app_max)

print("meniscal injuries after cleaning:", len(m))
print("...with both windows inside the appearances period (12 before, 24 after):", m["in_range"].sum())
print("...if only 12 months after were required (12 before, 12 after):", m["in_range_12"].sum())

def coverage(df):
    checks = {
        "appearances: match BEFORE":                  df["app_pre"],
        "appearances: match AFTER":                   df["app_post"],
        "appearances: BEFORE and AFTER":              df["app_pre"] & df["app_post"],
        "performances: season row BEFORE":            df["perf_pre"],
        "performances: season row AFTER":             df["perf_post"],
        "performances: BEFORE and AFTER":             df["perf_pre"] & df["perf_post"],
        "performances with minutes: BEFORE":          df["perf_pre_min"],
        "performances with minutes: AFTER":           df["perf_post_min"],
        "performances with minutes: BEFORE and AFTER": df["perf_pre_min"] & df["perf_post_min"],
    }
    return pd.DataFrame({"n": [v.sum() for v in checks.values()],
                         "%": [round(100 * v.mean(), 1) for v in checks.values()]},
                        index=list(checks.keys()))

r = m[m["in_range"]]
coverage(r)

meniscal injuries after cleaning: 1838
...with both windows inside the appearances period (12 before, 24 after): 1277
...if only 12 months after were required (12 before, 12 after): 1423


,n,%
appearances: match BEFORE,395,30.9
appearances: match AFTER,430,33.7
appearances: BEFORE and AFTER,320,25.1
performances: season row BEFORE,1214,95.1
performances: season row AFTER,1196,93.7
performances: BEFORE and AFTER,1142,89.4
performances with minutes: BEFORE,876,68.6
performances with minutes: AFTER,975,76.4
performances with minutes: BEFORE and AFTER,718,56.2


### 1.7 The same count for players of the top-5 leagues only


The table above covers **all** meniscal injuries in dataset 1, including players of lower leagues and from outside Europe. Our question is about the top 5, so we need a denominator: injuries of players who played in a match of one of the five leagues in the 12 months before the injury (`top5_pre`: at least `MIN_TOP5_MINUTES` minutes in `appearances`). The level is judged **only from the period before the injury**, so that players are not selected by what happens afterwards.

League ids in Transfermarkt: `GB1` (Premier League), `ES1` (LaLiga), `IT1` (Serie A), `L1` (Bundesliga), `FR1` (Ligue 1).

In [49]:
# check that these ids are really those leagues
player_performances.loc[player_performances["competition_id"].isin(TOP5)].groupby("competition_id")["competition_name"].first()

competition_id
ES1            LaLiga
FR1           Ligue 1
GB1    Premier League
IT1           Serie A
L1         Bundesliga
Name: competition_name, dtype: str

In [50]:

MIN_TOP5_MINUTES = 1          # at least this many minutes in top-5 league matches in the window before the injury

a5 = appearances.loc[appearances["player_id"].isin(m["player_id"]) & appearances["competition_id"].isin(TOP5),
                     ["player_id", "date", "minutes_played"]]
x5 = m[["inj_id", "player_id", "pre_start", "from_date"]].merge(a5, on="player_id")
x5 = x5[(x5["date"] >= x5["pre_start"]) & (x5["date"] < x5["from_date"])]
m["min_top5_pre"] = m["inj_id"].map(x5.groupby("inj_id")["minutes_played"].sum()).fillna(0)
m["top5_pre"] = m["min_top5_pre"] >= MIN_TOP5_MINUTES

r = m[m["in_range"]]
t5 = r[r["top5_pre"]]
print(f"injuries with windows in range: {len(r)}, of which the player played in a top-5 league in the 12 months before: {len(t5)}")

pd.concat({"all in range": coverage(r), "top 5 in the 12 months before": coverage(t5)}, axis=1)

injuries with windows in range: 1277, of which the player played in a top-5 league in the 12 months before: 197


all in range       top 5 in the 12 months before       
                                                       n     %                             n      %
appearances: match BEFORE                            395  30.9                           197  100.0
appearances: match AFTER                             430  33.7                           160   81.2
appearances: BEFORE and AFTER                        320  25.1                           160   81.2
performances: season row BEFORE                     1214  95.1                           196   99.5
performances: season row AFTER                      1196  93.7                           183   92.9
performances: BEFORE and AFTER                      1142  89.4                           182   92.4
performances with minutes: BEFORE                    876  68.6                           144   73.1
performances with minutes: AFTER                     975  76.4                           153   77.7
performances with minutes: BEFORE and AFTER          718  56.2                           127   64.5

Two remarks on this group: (1) "appearances: match BEFORE" is 100% here **by definition** (that is how we chose the group), so it says nothing about the data; (2) a player who did not play a top-5 match in the 12 months before the injury (for example a fresh signing, or a player who was out for a long time earlier) is not in this group.

### 1.8 Where are the players who have no match in `appearances` after the injury?

The group: injuries of top-5 players (`top5_pre`) with **no** match in `appearances` in the 24 months after the return. We check whether we can find them in `player_performances` (rows of the two seasons after the return season) and in `transfers`.

In [51]:
lost = m[m["in_range"] & m["top5_pre"] & ~m["app_post"]].copy()
print("injuries in this group:", len(lost), "| players:", lost["player_id"].nunique())

injuries in this group: 37 | players: 35


In [52]:
# player_performances rows from the two seasons after the return season
pp = player_performances.loc[player_performances["player_id"].isin(lost["player_id"]),
                             ["player_id", "season_start", "competition_id", "nb_on_pitch"]]
z = lost[["inj_id", "player_id", "ret_season"]].merge(pp, on="player_id")
z = z[(z["season_start"] > z["ret_season"]) & (z["season_start"] <= z["ret_season"] + 2)].copy()

z["is_top5_row"] = z["competition_id"].isin(TOP5)
z["top5_matches"] = np.where(z["is_top5_row"], z["nb_on_pitch"], 0)

agg = z.groupby("inj_id").agg(rows=("competition_id", "size"),
                              top5_rows=("is_top5_row", "sum"),
                              top5_matches=("top5_matches", "sum"))

lost = lost.merge(agg, left_on="inj_id", right_index=True, how="left")
lost[["rows", "top5_rows", "top5_matches"]] = lost[["rows", "top5_rows", "top5_matches"]].fillna(0)

lost["status_after"] = np.select(
    [lost["rows"] == 0, lost["top5_matches"] > 0, lost["top5_rows"] > 0],
    ["no row in performances", "plays in top 5 (missing from appearances)", "in a top-5 squad, 0 appearances"],
    default="plays outside the top 5")

lost["status_after"].value_counts()

status_after
plays outside the top 5                      24
no row in performances                        8
in a top-5 squad, 0 appearances               4
plays in top 5 (missing from appearances)     1
Name: count, dtype: int64

In [53]:
# those we cannot see in performances: in which season does their trace in that table end?
last_perf = player_performances.groupby("player_id")["season_start"].max()

gone = lost[lost["status_after"] == "no row in performances"]
gone.assign(last_season_in_performances=gone["player_id"].map(last_perf))[
    ["player_id", "from_date", "end_date", "last_season_in_performances"]]

,player_id,from_date,end_date,last_season_in_performances
474,22550,2016-09-15,2017-02-01,2016
479,22598,2019-02-10,2019-03-26,2018
830,34787,2023-04-29,2023-06-30,2022
1209,51426,2019-10-02,2020-01-09,2019
1236,524986,2022-01-24,2022-02-27,2021
1237,524986,2021-10-03,2021-11-03,2021
1250,5308,2018-02-19,2018-03-17,2017
1524,67089,2022-11-10,2023-06-10,2022


In [54]:
# transfers: do these injuries have a transfer between the injury start and the end of the window after?
t = transfers.loc[transfers["player_id"].isin(lost["player_id"]), ["player_id", "transfer_date", "to_club_name"]]
x2 = lost[["inj_id", "player_id", "from_date", "post_end"]].merge(t, on="player_id")
x2 = x2[(x2["transfer_date"] > x2["from_date"]) & (x2["transfer_date"] <= x2["post_end"])]

print("players of this group present in transfers at all:", lost["player_id"].drop_duplicates().isin(transfers["player_id"]).sum(),
      "of", lost["player_id"].nunique())
print("injuries with at least one transfer in the window:", x2["inj_id"].nunique(), "of", len(lost),
      "| of these transfers, to 'Retired' or 'Without Club':", int(x2["to_club_name"].isin(["Retired", "Without Club"]).sum()))

# does transfers mark the end of a career at all?
transfers.loc[transfers["to_club_name"].isin(["Retired", "Without Club", "Career break", "Unknown"]), "to_club_name"].value_counts()

players of this group present in transfers at all: 8 of 35
injuries with at least one transfer in the window: 5 of 37 | of these transfers, to 'Retired' or 'Without Club': 0


to_club_name
Without Club    3785
Unknown          268
Retired          184
Career break      63
Name: count, dtype: int64

### Findings

- **Meniscal injuries.** Five labels give 1,863 rows (1,657 players). 25 rows lack a start or end date and there are no full duplicates, so 1,838 injuries remain. Of the 1,657 players, 1,027 appear in `players`, 845 in `appearances` and all in `player_performances`.
- **Windows.** Both windows (12 months before, 24 after) lie inside the `appearances` period for 1,277 injuries (1,423 if only 12 months after were required). Only these are used below.
- **Coverage, all 1,277.** `appearances` has a match before the injury for 30.9%, after for 33.7% and both for 25.1%. `player_performances` has a season row before and after for 89.4% (56.2% with non-missing `minutes_played`).
- **Top-5 players** (at least 1 minute in a top-5 league match in the 12 months before): 197 of the 1,277 injuries. For them the data are much better: `appearances` has a match after the injury for 160 (81.2%), and `player_performances` has season rows before and after for 182 (92.4%).
- **The 37 top-5 injuries (35 players) without a match after.** According to `player_performances` 24 play outside the top 5, 4 are in a top-5 squad without an appearance, 1 plays in the top 5 (missing from `appearances`) and 8 have no row in the two seasons after. `transfers` does not explain them: 8 of the 35 players appear in it, 5 injuries have a transfer in the window and none goes to "Retired" or "Without Club".